# Too many tools

**Scenario:** a customer success assistant shipped with three tools. Two years and four teams later
it carries twenty four, and nobody removed one. A CSM asks whether an account is in trouble, and the
assistant makes a dozen backend calls to answer.

A tool list is **a menu with a hundred dishes**. Nobody orders better from a longer menu. They order
three things and hope one was right.

## Mechanics

The model picks from a list you send on every request. These fields decide what it picks and how
much runs.

| Field | Type | What it means |
|---|---|---|
| `tools` | list | Every function the model may choose, sent on every single request |
| `function.name` | string | The label it matches the request against |
| `function.description` | string | The only other thing it has to go on |
| `tool_choice` | string | `"required"` forces a tool call, and never says how many |
| `tool_calls` | list | What came back. Its length is what your backend pays for |

The last two are the trap. `tool_choice="required"` reads like "pick one". It means "pick at least
one", and nothing caps the list that comes back.

## The picture

![Confusable tool names spread one request across many backend calls](images/tool-entropy.svg)

Every extra name that could answer the question is another branch, and another call your backend may
be asked to run.

## The cost

Entropy is how spread out a choice is. Over a set of tools T, where P(t) is the share of answers
landing on tool t:

```
H(T)  = -sum P(t) log2 P(t)     bits of spread
calls = turns x calls per turn  what the backend runs
```

The formula is a way to think. The second line is the evidence.

## The failure

Three read tools, then the same three plus nine ordinary neighbours, then twelve near duplicates on
top. Every tool takes the same argument, so only the names and the words change.

In [1]:
ACCOUNT = {"type": "object", "properties": {"account_id": {"type": "string"}},
           "required": ["account_id"], "additionalProperties": False}


def reader(name, what):
    """One read-only tool. Identical shape every time, so only the wording differs."""
    return {"type": "function", "function": {
        "name": name, "description": f"Return {what} for one account.",
        "parameters": ACCOUNT}}


SMALL = [reader("get_account_health", "the live health score and risk band"),
         reader("get_usage_metrics", "product usage counters"),
         reader("get_contract_details", "contract terms, dates and committed spend")]

NEIGHBOURS = ["stakeholder_map", "open_tickets", "invoice_status", "nps_responses",
              "login_activity", "qbr_history", "billing_history", "success_plan",
              "renewal_forecast"]
NEAR_DUPLICATES = ["customer_health_summary", "account_risk_score", "churn_prediction",
                   "account_overview", "customer_snapshot", "health_trend", "account_summary",
                   "engagement_score", "product_adoption", "recent_activity",
                   "ticket_backlog", "account_notes"]

MID = SMALL + [reader(f"get_{n}", n.replace("_", " ")) for n in NEIGHBOURS]
BIG = MID + [reader(f"get_{n}", n.replace("_", " ")) for n in NEAR_DUPLICATES]
print(f"{len(SMALL)}, {len(MID)} and {len(BIG)} tools, and the same three jobs to do")

3, 12 and 24 tools, and the same three jobs to do


Three questions a CSM would really type. Each has one tool wired to a live system, and the system
prompt asks for one call.

In [2]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("10-low-entropy-tool-design/01-too-many-tools")

SYSTEM = "You are a customer success assistant. Answer by calling exactly one tool."
QUESTIONS = [("Is ACME-88 in trouble?", "get_account_health"),
             ("What has ACME-88 actually been using this quarter?", "get_usage_metrics"),
             ("When does the ACME-88 contract end?", "get_contract_details")]


def ask(tools, question):
    """One turn. Returns why generation stopped and every tool it asked for."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=900, tools=tools, tool_choice="required",
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": question}])
    choice = reply.choices[0]
    return choice.finish_reason, [c.function.name for c in (choice.message.tool_calls or [])]

Once is an anecdote, so each question is asked three times.

In [3]:
def trial(tools, repeats=3, wanted=None):
    """Ask all three questions a few times. Returns want, stop reason, and the ask."""
    return [(wanted or want, *ask(tools, question))
            for question, want in QUESTIONS for _ in range(repeats)]

Same nine questions, three tool sets. Only the length of the tool list changes.

In [4]:
runs = {"3 tools": trial(SMALL), "12 tools": trial(MID), "24 tools": trial(BIG)}

for label, rows in runs.items():
    calls = sum(len(got) for _, _, got in rows)
    wrong = sum(1 for want, _, got in rows if got[:1] != [want])
    refused = sum(1 for _, reason, _ in rows if reason != "tool_calls")
    print(f"{label:9} {calls:3} calls for {len(rows)} questions, "
          f"wrong tool {wrong}/{len(rows)}, generation refused {refused}/{len(rows)}")

worst = max(runs["24 tools"], key=lambda row: len(row[2]))
assert len(worst[2]) == 1, f"one turn asked for {len(worst[2])} calls, stopped on {worst[1]!r}"

3 tools     9 calls for 9 questions, wrong tool 0/9, generation refused 0/9
12 tools    9 calls for 9 questions, wrong tool 0/9, generation refused 0/9
24 tools  123 calls for 9 questions, wrong tool 3/9, generation refused 3/9


AssertionError: one turn asked for 39 calls, stopped on 'error'

## The diagnosis

Three tools and twelve tools behaved the same. Nine questions, nine calls, the wired tool every time.
At twenty four the same nine questions cost 123 calls.

**It did not pick badly. It stopped picking.** On "Is ACME-88 in trouble?" all twenty four names were
called, and `get_account_health` came back twenty two times inside one turn. The provider gave up on
that reply and returned `error` in place of `tool_calls`.

**The two sharp questions never wavered.** Usage and contract have one obvious owner in every set. It
is not how many tools you registered. It is how many could answer the request in front of you.

**`tool_choice="required"` capped nothing.** It asks for at least one call. It never asks for one.

Now put a number on the spread, using the entropy formula from the cost beat.

In [5]:
import math
from collections import Counter


def spread(rows, wanted):
    """Shannon entropy of where the calls landed, in bits."""
    landed = Counter(name for want, _, got in rows if want == wanted for name in got)
    total = sum(landed.values())
    bits = -sum((n / total) * math.log2(n / total) for n in landed.values())
    return abs(bits), len(landed)


for label, rows in runs.items():
    bits, names = spread(rows, "get_account_health")
    print(f"{label:9} 'Is ACME-88 in trouble?' landed on {names:2} names, {bits:.2f} bits")

3 tools   'Is ACME-88 in trouble?' landed on  1 names, 0.00 bits
12 tools  'Is ACME-88 in trouble?' landed on  1 names, 0.00 bits
24 tools  'Is ACME-88 in trouble?' landed on 24 names, 3.84 bits


## The fix

The fix is not a better name for each tool. It is fewer tools. The health family is one question
asked five ways, so it collapses into one tool with a closed list of signals as an argument. A schema
is a written shape the provider checks before your code sees anything. A sibling tool is not.

In [6]:
def scoped(name, description, extra):
    """A tool whose choices live in its arguments, not in its siblings."""
    props = {"account_id": {"type": "string"}, **extra}
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}}


SIGNALS = ["health", "usage", "contract", "tickets", "adoption"]
SCOPED = [scoped("get_account_signal", "Return one named signal for an account.",
                 {"signal": {"type": "string", "enum": SIGNALS}}),
          scoped("create_renewal_task", "Create a renewal review task for the CSM.",
                 {"due_date": {"type": "string"}}),
          scoped("log_interaction", "Record a conversation on the account timeline.",
                 {"channel": {"type": "string"}, "summary": {"type": "string"}})]

scoped_rows = trial(SCOPED, wanted="get_account_signal")
before, after = runs["24 tools"], scoped_rows
missed = [sum(1 for want, _, got in rows if got[:1] != [want]) for rows in (before, after)]
calls = [sum(len(got) for _, _, got in rows) for rows in (before, after)]
bits = [spread(before, "get_account_health")[0], spread(after, "get_account_signal")[0]]
print(f"backend calls  24 tools: {calls[0]:3}     3 tools: {calls[1]:3}")
print(f"bits of spread 24 tools: {bits[0]:.2f}    3 tools: {bits[1]:.2f}")
print(f"wrong tool     24 tools: {missed[0]}/9     3 tools: {missed[1]}/9")

backend calls  24 tools: 123     3 tools:   9
bits of spread 24 tools: 3.84    3 tools: 0.00
wrong tool     24 tools: 3/9     3 tools: 0/9


Three questions, three calls each, the right signal every time, and the spread is back to zero bits.
The design cannot promise how many calls one turn may run. The ordinary code you write around the
model decides that, for both tool sets.

In [7]:
def cap_reads(picked, limit=1):
    """The harness decides how many reads one turn may run. The model does not."""
    return picked[:limit], max(0, len(picked) - limit)


for label, rows in (("24 tools", before), ("3 tools", after)):
    ran = sum(len(cap_reads(got)[0]) for _, _, got in rows)
    held = sum(cap_reads(got)[1] for _, _, got in rows)
    print(f"{label:9} one read a turn: {ran:3} ran, {held:3} refused and written down")

24 tools  one read a turn:   9 ran, 114 refused and written down
3 tools   one read a turn:   9 ran,   0 refused and written down


## The gate

The cap held the bloated set to nine calls too, and its first call was still wrong three times in
nine. A cap bounds the damage. It does not choose better.

The regression to stop is a sixth read tool answering a question an existing tool already answers. So
the catalogue records what each tool returns, and the check rejects two owners for one answer.

In [8]:
CATALOGUE = {"get_account_signal": set(SIGNALS),
             "create_renewal_task": {"renewal_task"},
             "log_interaction": {"timeline_note"}}


def test_one_tool_per_answer():
    owners = Counter(field for fields in CATALOGUE.values() for field in fields)
    shared = sorted(field for field, count in owners.items() if count > 1)
    assert not shared, f"more than one tool answers for {shared}"


test_one_tool_per_answer()
print("gate holds: every answer in the catalogue has exactly one tool that returns it")

gate holds: every answer in the catalogue has exactly one tool that returns it


Add `"get_health_trend": {"health"}` to the catalogue and this test fails before anyone ships it.

### Enterprise exploration

- Tool schemas ride on every request. At what volume does three tools against twenty four show up on
  the bill, and how would you measure it?
- Twelve reads in one turn is twelve times the load on the account service. What is your rate limit,
  and does it protect the backend or only the agent?
- Collapsing five tools into one argument breaks every caller. How do you ship that without an
  outage, and what is the compliance record of the old calls?

### Key takeaways

- `tool_choice="required"` means at least one call, never exactly one.
- Confusable names cost twice. The model picks a near duplicate, then asks for the rest as well.
- A closed list inside one tool is checked. Five sibling tools are not.
- Entropy is a way to think about spread. The call count is the evidence.